# Titanic Survival Prediction — Data Preprocessing

## Objective

Prepare the Titanic dataset for supervised machine learning.

Steps:
1. Select input features and target.
2. Split the labeled data into training and validation sets.
3. Handle missing values.
4. Encode categorical variables.
5. Scale numerical features.
6. Prepare the data for model training.

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/raw/train.csv")

train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = train_df[features]
y = train_df["Survived"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (891, 7)
Target shape: (891,)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (712, 7)
X_val: (179, 7)
y_train: (712,)
y_val: (179,)


In [4]:
print("Training missing values:")
print(X_train.isnull().sum())

print("\nValidation missing values:")
print(X_val.isnull().sum())

Training missing values:
Pclass        0
Sex           0
Age         137
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

Validation missing values:
Pclass       0
Sex          0
Age         40
SibSp        0
Parch        0
Fare         0
Embarked     0
dtype: int64


In [5]:
from sklearn.impute import SimpleImputer

age_imputer = SimpleImputer(strategy="median")

X_train = X_train.copy()
X_val = X_val.copy()

X_train["Age"] = age_imputer.fit_transform(
    X_train[["Age"]]
).ravel()

X_val["Age"] = age_imputer.transform(
    X_val[["Age"]]
).ravel()

In [6]:
embarked_imputer = SimpleImputer(strategy="most_frequent")

X_train["Embarked"] = embarked_imputer.fit_transform(
    X_train[["Embarked"]]
).ravel()

X_val["Embarked"] = embarked_imputer.transform(
    X_val[["Embarked"]]
).ravel()

In [7]:
print("Training missing values:")
print(X_train.isnull().sum())

print("\nValidation missing values:")
print(X_val.isnull().sum())

Training missing values:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

Validation missing values:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


## Encoding Categorical Features

We use One-Hot Encoding to convert categorical variables
into numerical representations.

The encoder is fitted only on the training dataset
to prevent data leakage.

In [8]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["Sex", "Embarked"]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_train = encoder.fit_transform(
    X_train[categorical_features]
)

encoded_val = encoder.transform(
    X_val[categorical_features]
)

print("Encoded training shape:", encoded_train.shape)
print("Encoded validation shape:", encoded_val.shape)

Encoded training shape: (712, 5)
Encoded validation shape: (179, 5)


In [9]:
encoded_columns = encoder.get_feature_names_out(
    categorical_features
)

print(encoded_columns)

['Sex_female' 'Sex_male' 'Embarked_C' 'Embarked_Q' 'Embarked_S']


In [10]:
encoded_columns = encoder.get_feature_names_out(
    categorical_features
)

print(encoded_columns)

['Sex_female' 'Sex_male' 'Embarked_C' 'Embarked_Q' 'Embarked_S']


## Feature Scaling

StandardScaler is used to standardize numerical features.

The scaler learns the mean and standard deviation
from the training dataset only, preventing data leakage.


In [11]:
from sklearn.preprocessing import StandardScaler

numerical_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

scaler = StandardScaler()

scaled_train = scaler.fit_transform(
    X_train[numerical_features]
)

scaled_val = scaler.transform(
    X_val[numerical_features]
)

print("Scaled training shape:", scaled_train.shape)
print("Scaled validation shape:", scaled_val.shape)

Scaled training shape: (712, 5)
Scaled validation shape: (179, 5)


In [12]:
scaled_train_df = pd.DataFrame(
    scaled_train,
    columns=numerical_features,
    index=X_train.index
)

scaled_train_df.head()

,Pclass,Age,SibSp,Parch,Fare
692,0.829568,-0.081135,-0.465084,-0.466183,0.513812
481,-0.370945,-0.081135,-0.465084,-0.466183,-0.662563
527,-1.571457,-0.081135,-0.465084,-0.466183,3.955399
855,0.829568,-0.887827,-0.465084,0.727782,-0.467874
801,-0.370945,0.110934,0.478335,0.727782,-0.115977


## Building the Preprocessing Pipeline

We combine missing-value imputation, categorical encoding,
and numerical scaling into reusable Scikit-learn pipelines.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numerical_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Sex",
    "Embarked"
]

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [15]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [17]:
X = train_df[features]
y = train_df["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)

Processed training shape: (712, 10)
Processed validation shape: (179, 10)


In [19]:
processed_columns = preprocessor.get_feature_names_out()

processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=processed_columns,
    index=X_train.index
)

processed_train_df.head()

,num__Pclass,num__Age,num__SibSp,num__Parch,num__Fare,cat__Sex_female,cat__Sex_male,cat__Embarked_C,cat__Embarked_Q,cat__Embarked_S
692,0.829568,-0.081135,-0.465084,-0.466183,0.513812,0.0,1.0,0.0,0.0,1.0
481,-0.370945,-0.081135,-0.465084,-0.466183,-0.662563,0.0,1.0,0.0,0.0,1.0
527,-1.571457,-0.081135,-0.465084,-0.466183,3.955399,0.0,1.0,0.0,0.0,1.0
855,0.829568,-0.887827,-0.465084,0.727782,-0.467874,1.0,0.0,0.0,0.0,1.0
801,-0.370945,0.110934,0.478335,0.727782,-0.115977,1.0,0.0,0.0,0.0,1.0


In [20]:
print("Missing values:", processed_train_df.isnull().sum().sum())

Missing values: 0
